In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm import tqdm
import opensmile
from python_speech_features import mfcc, logfbank
import parselmouth
from scipy.stats import skew, kurtosis
import soundfile as sf
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import speech_recognition as sr
from concurrent.futures import ThreadPoolExecutor, as_completed # Añade concurrencia, importada para acelerar la ejecución
import time, random
from pathlib import Path

from vosk import Model, KaldiRecognizer
import numpy as np
import json

#Es un modelo a descargar de https://alphacephei.com/vosk/models, hay varios idiomas, por ahora solo he usado inglés.
MODEL_DIR = r"C:\Users\gerbq\OneDrive\Escritorio\TFG\vosk-model-small-en-us-0.15"  
vosk_model = Model(MODEL_DIR)


In [2]:
def crear_dataset_binario(ruta_nat: str,
                          ruta_gen: str,
                          extension: str = "*.wav",
                          random_state: int = 42) -> pd.DataFrame:
    """
    Crea un DataFrame con rutas de audio y etiquetas:
    - 'nat' para audios en ruta_natural
    - 'gen' para audios en ruta_generado
    """

    # Rutas de audios reales 
    paths_nat = [str(p) for p in Path(ruta_nat).rglob(extension)]

    # Rutas de audios generados
    paths_gen = [str(p) for p in Path(ruta_gen).rglob(extension)]

    df = pd.DataFrame({
        "path": paths_nat + paths_gen,
        "label": (["nat"] * len(paths_nat)) + (["gen"] * len(paths_gen))
    })

    return df

In [3]:
ruta_nat= r"C:\Users\gerbq\OneDrive\Escritorio\TFG\Datasets\Propio\Natural\Inglés"
ruta_gen    = r"C:\Users\gerbq\OneDrive\Escritorio\TFG\Datasets\Propio\Generado\Inglés"

df_dataset = crear_dataset_binario(ruta_nat, ruta_gen)

print(df_dataset.head())


                                                path label
0  C:\Users\gerbq\OneDrive\Escritorio\TFG\Dataset...   nat
1  C:\Users\gerbq\OneDrive\Escritorio\TFG\Dataset...   nat
2  C:\Users\gerbq\OneDrive\Escritorio\TFG\Dataset...   nat
3  C:\Users\gerbq\OneDrive\Escritorio\TFG\Dataset...   nat
4  C:\Users\gerbq\OneDrive\Escritorio\TFG\Dataset...   nat


In [15]:
from pyexpat import features


def extract_paraling_features(y, sr): # y: muestras de audio, sr: frecuencia de muestreo
    features = {}
    features["RMS"] = np.mean(librosa.feature.rms(y = y)) 
    features["ZCR"] = np.mean(librosa.feature.zero_crossing_rate(y)) 
    features["Centroid"] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    features["Bandwidth"] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    features["Rolloff"] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    mfcc_mean = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
    features["MFCC_mean"] = "[" + ", ".join(f"{x:.4f}" for x in mfcc_mean) + "]"
    features["Pitch"] = np.mean(librosa.yin(y, fmin=50, fmax=400))
    return features

In [16]:
def transcribir_audio(path):
    """
    Transcribe un audio usando Vosk.
    - Convierte a mono 16 kHz internamente.
    - Devuelve el texto reconocido (string).
    """
    # 1) Cargar audio como mono 16 kHz
    y, sr = librosa.load(path, sr=16000, mono=True)

    # 2) Pasar de float32 [-1, 1] a int16
    y_int16 = (y * 32767).astype(np.int16)
    data = y_int16.tobytes()

    # 3) Crear reconocedor
    rec = KaldiRecognizer(vosk_model, 16000)
    rec.SetWords(True)

    # 4) Alimentar por chunks
    chunk_size = 4000 * 2  # 4000 muestras * 2 bytes (int16)
    texto = ""

    for i in range(0, len(data), chunk_size):
        chunk = data[i : i + chunk_size]
        if rec.AcceptWaveform(chunk):
            res = json.loads(rec.Result())
            texto += " " + res.get("text", "")

    res_final = json.loads(rec.FinalResult())
    texto += " " + res_final.get("text", "")

    return texto.strip()


In [17]:
def cargar_audios_en_df(audios, extensiones=(".wav"), sr=None, max_workers=8, labels=None):

    labels_map = None
    if labels is not None:
        labels_map = {path: lab for path, lab in zip(audios, labels)}

    def cargar_un_audio(path):
        try:
            features = {}
            features.update({"Audio": os.path.splitext(os.path.basename(path))[0]})
            features["Idioma"] = "Inglés"
            if labels_map is not None:
                features["Tipo"] = labels_map.get(path, None)

            y, sr_local = librosa.load(path, sr=sr)
            duracion = librosa.get_duration(y=y, sr=sr_local)
            
            texto = transcribir_audio(path)
            features["Transcription"] = texto

            features.update({"Duracion": duracion})
            features.update(extract_paraling_features(y, sr_local))

            
            return features

        except Exception as e:
            d = {
                "archivo": os.path.basename(path),
                "ruta": path,
                "error": str(e)
            }
            if labels_map is not None:
                d["label"] = labels_map.get(path, None)
            return d

    resultados = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {executor.submit(cargar_un_audio, path): path for path in audios}
        
        for futuro in tqdm(as_completed(futuros), total=len(futuros), desc="Cargando audios", ncols=100):
            resultados.append(futuro.result())

    df = pd.DataFrame(resultados)
    print("✅ Carga completada.")
    return df


In [18]:
df_features = cargar_audios_en_df(
    audios=df_dataset["path"].tolist(),
    labels=df_dataset["label"].tolist(),
    sr=16000
)

print(df_features.head())

Cargando audios:   0%|                                                      | 0/150 [00:00<?, ?it/s]

Cargando audios: 100%|████████████████████████████████████████████| 150/150 [00:41<00:00,  3.59it/s]

✅ Carga completada.
    Audio  Idioma Tipo                Transcription  Duracion       RMS  \
0  I_14_N  Inglés  nat                what's it for  2.838938  0.022534   
1  I_12_N  Inglés  nat                 what is next  2.264813  0.019565   
2  I_10_N  Inglés  nat         the par is up for it  2.666937  0.030686   
3  I_15_N  Inglés  nat  they may have been murdered  2.646313  0.035857   
4  I_13_N  Inglés  nat   horses are drawn by ballot  2.862188  0.038890   

        ZCR     Centroid    Bandwidth      Rolloff  \
0  0.061414   941.164852  1327.304120  1498.683287   
1  0.097704  1636.524177  1590.950084  2916.263204   
2  0.049776  1093.574429  1484.859277  2176.339286   
3  0.078143  1456.546366  1467.164634  2707.454819   
4  0.103950  1582.696875  1556.154364  2952.604167   

                                           MFCC_mean       Pitch  
0  [-456.3004, 65.6086, 11.9345, 5.8275, -4.3167,...  148.816934  
1  [-456.7985, 45.5414, 2.0627, 12.1647, -1.4552,...  174.586992  
2  

In [ ]:
df_features.to_csv("Audios_features.csv", index=False, float_format="%.4f", encoding="utf-8")